# tiny-voca -> Roman Urdu (1000 UrduTTS clips)
Kaggle: turn **Internet ON** and pick a **GPU** accelerator. Run top to bottom.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: set Settings -> Accelerator -> GPU (before Save Version too)'

In [ ]:
STEPS = 25000     # fine-tune steps (~3 h on a T4)
N_TRAIN = 5000    # ~8 h of audio; whole run ~8-8.5 h
N_VAL = 64

In [ ]:
%%bash
set -e
mkdir -p /kaggle/working/tiny-urdu && cd /kaggle/working/tiny-urdu
[ -d tiny-voca ] || git clone -q https://github.com/n4ze3m/tiny-voca
git -C tiny-voca checkout -q ca93ad1e35a834a9247347c6821cfeb4c82c6a96
pip install -q --no-deps -e tiny-voca
# one consistent set: transformers<5 needs huggingface-hub<1, so datasets must be <4
pip install -q "huggingface-hub>=0.34,<1" "transformers>=4.40,<5" "datasets<4" librosa pyarrow scipy soundfile 2>&1 | grep -v -i -E "dependency resolver|requires|^$" || true
python - <<'PY'
from huggingface_hub import hf_hub_download, snapshot_download
snapshot_download("facebook/wav2vec2-lv-60-espeak-cv-ft", allow_patterns=["*.json", "pytorch_model.bin", "*.txt"])
hf_hub_download("n4ze3m/tiny-voca", "tiny-voca.pt", local_dir="weights")
import datasets, transformers, tiny_voca; print("imports ok", datasets.__version__, transformers.__version__)
PY

In [ ]:
%cd /kaggle/working/tiny-urdu
!mkdir -p urdu runs

## Urdu front end + helper scripts

In [ ]:
%%writefile urdu/frontend.py
"""Urdu front end for tiny-voca.

tiny-voca's 76 English phoneme symbols are kept (so the English checkpoint can
be fine-tuned) and NATIVE Urdu phones are appended: retroflex ʈ ɖ ɽ, aspirated
kʰ tʰ bʰ ..., q x ɣ ʋ, long/nasal vowels aː eː ẽ ã ... Every symbol is a label
of the wav2vec2 forced aligner, spelled exactly as in its vocab.json (nasal
vowels decomposed: base + U+0303). Mapping Urdu onto English symbols instead
(kʰ -> k h, ʈ -> t) made the aligner reject 998/1000 clips.

Input to ``phonemize``:
* ``"ipa:<phonemes>"``  the UrduTTS ``phonemes`` column (TRAINING)
* anything else         Roman Urdu, e.g. ``"aap kaise hain?"`` (INFERENCE)
  Capital T, D, R mark retroflex ʈ ɖ ɽ (``baRa`` = big, ``bura`` = bad).

Both routes go through ``_ipa_word``, so they produce one token language.
Stress marks are dropped (Roman Urdu has none to reproduce).
"""

import hashlib
import json
import re
import unicodedata

IPA_PREFIX = "ipa:"
VERSION = "urdu-2"
N = "̃"  # combining tilde (nasal)

# Appended to tiny-voca's SYMBOLS. APPEND-ONLY: reordering breaks checkpoints.
URDU_CONSONANTS = ("ʈ", "ɖ", "ɽ", "q", "x", "ɣ", "ʋ", "r", "ɟ", "c", "ʂ", "ʐ", "ɳ", "ɲ", "ɭ",
                   "kʰ", "pʰ", "tʰ", "ʈʰ", "bʰ", "dʰ", "ɖʰ", "ɡʰ", "ɟʰ", "cʰ")
URDU_VOWELS = ("a", "aː", "e", "eː", "o", "u", "ɛː", "ɪː", "ʊː",
               "a" + N, "e" + N, "e" + N + "ː", "o" + N, "i" + N, "u" + N, "ɛ" + N, "ɔ" + N, "ɐ" + N, "ɑ" + N)
URDU_PHONES = URDU_CONSONANTS + URDU_VOWELS
URDU_VOICELESS = ("ʈ", "q", "x", "c", "ʂ", "kʰ", "pʰ", "tʰ", "ʈʰ", "cʰ")

_IGNORED = set("ˈˌ.|‿͜͡")
# Last resort when no native symbol exists (rare in the data).
_APPROX = {"ɹ": "ɹ", "ɡ": "ɡ", "g": "ɡ", "ə": "ə", "ʌ": "ʌ", "ɐ": "ɐ", "ɑ": "ɑː", "ɔ": "ɔ", "ʊ": "ʊ", "ɪ": "ɪ", "i": "i", "ɛ": "ɛ"}
_NASAL_BASE = {"ə": "ɐ", "ʌ": "ɐ", "ɐ": "ɐ", "ɪ": "i", "ʊ": "u", "ɑ": "ɑ"}

_inventory = None   # set by install(): the full SYMBOLS phone set


def _resolve(base, long, aspirated, nasal, inventory):
    """One IPA segment -> list of inventory symbols, preferring the native one."""
    L, H = ("ː" if long else ""), ("ʰ" if aspirated else "")
    base = "ɡ" if base == "g" else base
    candidates = []
    if nasal:
        for b in (base, _NASAL_BASE.get(base, base)):
            candidates += [b + N + L, b + N]
    candidates += [base + H + L, base + H]
    if not H and not L:
        candidates.append(_APPROX.get(base, base))
    if long:
        candidates += [_APPROX.get(base, base) + "ː", {"ɑ": "ɑː"}.get(base, "")]
    for c in candidates:
        if c in inventory:
            return [c]
    if aspirated:                                  # e.g. ɽʰ -> ɽ h
        return _resolve(base, long, False, nasal, inventory) + ["h"]
    return [base]                                  # unknown -> <unk>, reported by the annotator


def _ipa_word(word, inventory=None):
    inventory = inventory or _inventory
    units = []
    for char in unicodedata.normalize("NFD", word):
        if char in _IGNORED:
            continue
        if char in "ː:" and units:
            units[-1][1] = True
        elif char == "ʰ" and units:
            units[-1][2] = True
        elif char == N and units:
            units[-1][3] = True
        elif unicodedata.combining(char):
            continue
        else:
            units.append([char, False, False, False])
    phones = []
    for base, long, aspirated, nasal in units:
        phones += _resolve(base, long, aspirated, nasal, inventory)
    return phones


# ---- Roman Urdu -> IPA string -> _ipa_word --------------------------------------------------

_LEXICON = {   # Roman spelling -> IPA as the dataset writes it
    "hai": "hɛ", "hay": "hɛ", "he": "hɛ", "hain": "hẽ", "hein": "hẽ", "hen": "hẽ",
    "mein": "mẽ", "main": "mẽ", "men": "mẽ", "me": "mẽ", "nahi": "nahiː", "nahin": "nahĩ", "nahee": "nahiː",
    "aur": "ɔr", "or": "ɔr", "ka": "kaː", "ki": "ki", "ke": "keː", "ko": "koː", "se": "seː", "ne": "neː",
    "par": "par", "pe": "peː", "yeh": "jeː", "ye": "jeː", "woh": "ʋoː", "wo": "ʋoː", "to": "toː", "bhi": "bʰiː",
    "hi": "hiː", "is": "ɪs", "us": "ʊs", "in": "ɪn", "un": "ʊn", "kya": "kjaː", "kyun": "kjũ", "kyon": "kjũ",
    "kuch": "kʊcʰ", "kuchh": "kʊcʰ", "tha": "tʰaː", "thi": "tʰiː", "the": "tʰeː", "thay": "tʰeː", "ho": "hoː",
    "hoon": "hũ", "hun": "hũ", "hum": "ham", "tum": "tʊm", "aap": "aːp", "ap": "aːp", "ek": "eːk", "do": "doː",
    "jo": "ɟoː", "jab": "ɟab", "tak": "tak", "allah": "allaːh", "mera": "meːraː", "meri": "meːriː",
    "mere": "meːreː", "apna": "apnaː", "apni": "apniː", "apne": "apneː", "sab": "sab", "ab": "ab",
}
# Extra spellings added after training. Kept OUT of _rules_digest so trained
# checkpoints still load (infer.py checks the frontend fingerprint). Training
# uses the ipa: path, so this only affects Roman input. Add words here freely.
_LEXICON_EXTRA = {
    "chai": "caːeː", "chaye": "caːeː", "chae": "caːeː", "chaaye": "caːeː", "chaey": "caːeː",
}
_RULES = (   # longest first; output is IPA
    ("chh", "cʰ"), ("Th", "ʈʰ"), ("Dh", "ɖʰ"), ("Rh", "ɽʰ"),
    ("kh", "kʰ"), ("gh", "ɡʰ"), ("ch", "c"), ("sh", "ʃ"), ("th", "tʰ"), ("dh", "dʰ"), ("ph", "pʰ"),
    ("bh", "bʰ"), ("jh", "ɟʰ"), ("zh", "ʒ"), ("ck", "k"),
    ("aa", "aː"), ("ee", "iː"), ("ii", "iː"), ("oo", "uː"), ("ai", "ɛ"), ("ay", "eː"), ("ey", "eː"),
    ("ei", "ɛ"), ("au", "ɔ"), ("ou", "ɔ"), ("ao", "aːoː"),
    ("T", "ʈ"), ("D", "ɖ"), ("R", "ɽ"),
)
_SINGLE = {"a": "a", "e": "eː", "i": "ɪ", "o": "oː", "u": "ʊ", "g": "ɡ", "j": "ɟ", "y": "j", "w": "ʋ",
           "v": "ʋ", "c": "k", "x": "ks", "r": "r"}
_VOWEL_LETTERS = set("aeiou")


def roman_to_ipa(word):
    if word.lower() in _LEXICON_EXTRA and not any(c in "TDR" for c in word[1:]):
        return _LEXICON_EXTRA[word.lower()]
    if word.lower() in _LEXICON and not any(c in "TDR" for c in word):
        return _LEXICON[word.lower()]
    # Capital T/D/R inside a word mark retroflex. A capital first letter is just
    # sentence case, and ALL CAPS words are read as plain lowercase.
    if word.isupper() and len(word) > 1:
        word = word.lower()
    word = word[:1].lower() + word[1:]
    if word.lower() in _LEXICON and not any(c in "TDR" for c in word):
        return _LEXICON[word.lower()]
    word = "".join(c if c in "TDR" else c.lower() for c in word.replace("'", ""))
    out, i, last = [], 0, ""
    while i < len(word):
        for letters, ipa in _RULES:
            if word.startswith(letters, i):
                out.append(ipa)
                i += len(letters)
                break
        else:
            ch, final = word[i], i == len(word) - 1
            if ch == "a" and final and len(word) > 1:
                ipa = "aː"
            elif ch == "i" and final and len(word) > 1:
                ipa = "iː"
            else:
                ipa = _SINGLE.get(ch, ch if ch.isalpha() else "")
            if ipa and not (ch not in _VOWEL_LETTERS and ch == last):   # "achha": say a doubled consonant once
                out.append(ipa)
            last = ch
            i += 1
            continue
        last = ""
    return "".join(out)


_TOKEN = re.compile(r"[A-Za-z']+|[,.?!;:\"()—…]")


def pieces(text):
    if text.startswith(IPA_PREFIX):
        words = [w for w in (_ipa_word(w) for w in text[len(IPA_PREFIX):].split()) if w]
        return [(False, [[(p, 0) for p in w] for w in words])]
    from tiny_voca.text import normalize_text
    result, words = [], []
    for token in _TOKEN.findall(normalize_text(text).replace("-", " ")):
        if token[0].isalpha() or token[0] == "'":
            phones = _ipa_word(roman_to_ipa(token))
            if phones:
                words.append([(p, 0) for p in phones])
        else:
            if words:
                result.append((False, words))
                words = []
            result.append((True, token))
    if words:
        result.append((False, words))
    return result


def _rules_digest():
    blob = json.dumps([URDU_PHONES, _LEXICON, _RULES, _SINGLE, _APPROX, VERSION], ensure_ascii=False)
    return hashlib.sha256(blob.encode()).hexdigest()[:16]


# ---- tiny-voca integration -----------------------------------------------------------------

def install():
    """Extend tiny_voca's symbol inventory with Urdu phones and swap in this front
    end. Must run BEFORE tiny_voca.{data,align,annotate,train,infer} are imported."""
    global _inventory
    import dataclasses
    import importlib
    import tiny_voca
    from tiny_voca import config as config_module, text

    if getattr(text, "_urdu_installed", False):
        return
    english = text.SYMBOLS
    assert len(english) == 76 and not set(URDU_PHONES) & set(english)
    text.CONSONANTS = text.CONSONANTS + URDU_CONSONANTS
    text.VOWELS = text.VOWELS + URDU_VOWELS
    text.PHONES = text.PHONES + URDU_PHONES
    text.SYMBOLS = english + URDU_PHONES          # English ids unchanged; Urdu appended
    text.SYMBOL_TO_ID = {s: i for i, s in enumerate(text.SYMBOLS)}
    text.PHONE_SET = frozenset(text.PHONES)
    text.PHONE_IDS = frozenset(text.SYMBOL_TO_ID[p] for p in text.PHONES) | {text.SYMBOL_TO_ID[text.UNK]}
    text.VOICELESS = text.VOICELESS | frozenset(URDU_VOICELESS)
    text.VOWEL_SET = frozenset(text.VOWELS)
    _inventory = text.PHONE_SET

    def phonemize(value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError("text must not be empty")
        return text.tokenize(pieces(value))

    def frontend_identity():
        return {"frontend_version": VERSION, "phonemizer": "tiny-urdu", "rules": _rules_digest(), "symbols": list(text.SYMBOLS)}

    text.phonemize = phonemize
    text.frontend_identity = frontend_identity
    text.frontend_runtime = lambda: {"phonemizer": "tiny-urdu-" + _rules_digest()}

    # Config: vocab_size default follows the larger inventory.
    UrduConfig = dataclasses.make_dataclass(
        "Config", [("vocab_size", int, dataclasses.field(default=len(text.SYMBOLS)))],
        bases=(config_module.Config,), frozen=True)
    UrduConfig.__module__ = config_module.__name__
    config_module.Config = UrduConfig
    tiny_voca.Config = UrduConfig
    # model.py was imported by tiny_voca/__init__ with the old symbol sets: re-import it.
    model = importlib.reload(importlib.import_module("tiny_voca.model"))
    tiny_voca.TinyVoca = model.TinyVoca
    text._urdu_installed = True

In [ ]:
%%writefile urdu/run.py
"""Run a tiny_voca module with the Urdu front end installed.

    python urdu/run.py tiny_voca.annotate --features ...
    python urdu/run.py tiny_voca.train ...
    python urdu/run.py tiny_voca.infer --checkpoint ... --text "aap kaise hain?"

The patch has to happen before the target module is imported, because
tiny-voca modules do ``from .text import phonemize``.
"""

import os
import runpy
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import frontend  # noqa: E402

frontend.install()

if __name__ == "__main__":
    if len(sys.argv) < 2:
        raise SystemExit(__doc__)
    module = sys.argv[1]
    sys.argv = [module] + sys.argv[2:]
    runpy.run_module(module, run_name="__main__", alter_sys=True)

In [ ]:
%%writefile urdu/prepare_data.py
"""Stream N UrduTTS clips, resample 44.1k -> 24k, write wavs + tiny-voca JSONL.

    python urdu/prepare_data.py --train 1000 --validation 32 --output data

Training text is "ipa:" + the dataset's `phonemes` column (see urdu/frontend.py
for why the vowel-less `roman` column is not used).
"""

import argparse
import json
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly

SR = 24000
SOURCE_SR = 44100
REPO = "kaab4321/UrduTTS"


def stream(split, count, min_s, max_s, out_dir, speaker, prefix):
    from datasets import Audio, load_dataset
    ds = load_dataset(REPO, split=split, streaming=True).cast_column("audio", Audio(decode=False))
    wav_dir = out_dir / "wavs"
    wav_dir.mkdir(parents=True, exist_ok=True)
    rows, skipped = [], 0
    for index, row in enumerate(ds):
        if len(rows) >= count:
            break
        phonemes = (row.get("phonemes") or "").strip()
        if not phonemes:
            skipped += 1
            continue
        wave, sr = sf.read(__import__("io").BytesIO(row["audio"]["bytes"]), dtype="float32")
        if wave.ndim > 1:
            wave = wave.mean(axis=1)
        seconds = len(wave) / sr
        if not min_s <= seconds <= max_s:
            skipped += 1
            continue
        if sr != SR:
            g = np.gcd(sr, SR)
            wave = resample_poly(wave, SR // g, sr // g).astype(np.float32)
        peak = float(np.max(np.abs(wave)))
        if peak > 0.99:
            wave = wave * (0.99 / peak)
        name = f"{prefix}{len(rows):05d}"
        sf.write(wav_dir / f"{name}.wav", wave, SR, subtype="PCM_16")
        rows.append({"id": name, "text": "ipa:" + phonemes, "speaker": speaker, "audio": f"wavs/{name}.wav",
                     "origin": "kaab4321/UrduTTS", "source_id": f"{split}:{index}",
                     "urdu": row.get("transcription", "").strip(), "seconds": round(seconds, 2)})
        if len(rows) % 100 == 0:
            print(f"[{split}] {len(rows)}/{count} (skipped {skipped})", flush=True)
    return rows


def median_f0(out_dir, rows, n=20):
    import librosa
    f0s = []
    for row in rows[:n]:
        wave, _ = sf.read(out_dir / row["audio"], dtype="float32")
        f0 = librosa.yin(wave, fmin=60, fmax=500, sr=SR, frame_length=2048)
        rms = librosa.feature.rms(y=wave, frame_length=2048, hop_length=512)[0][:len(f0)]
        f0s.extend(f0[:len(rms)][rms > np.percentile(rms, 60)])
    return float(np.median(f0s))


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--train", type=int, default=1000)
    p.add_argument("--validation", type=int, default=32)
    p.add_argument("--output", type=Path, default=Path("data"))
    p.add_argument("--min-seconds", type=float, default=1.5)
    p.add_argument("--max-seconds", type=float, default=12.0, help="shorter clips = cheaper CPU training")
    p.add_argument("--speaker", choices=("auto", "F2", "M2"), default="auto",
                   help="which tiny-voca voice slot the Urdu speaker is written into (auto = by median pitch)")
    a = p.parse_args()
    a.output.mkdir(parents=True, exist_ok=True)

    train = stream("train", a.train, a.min_seconds, a.max_seconds, a.output, "F2", "tr")
    speaker = a.speaker
    if speaker == "auto":
        f0 = median_f0(a.output, train)
        speaker = "F2" if f0 > 165 else "M2"
        print(f"median f0 {f0:.0f} Hz -> voice slot {speaker}")
    val = stream("test", a.validation, a.min_seconds, a.max_seconds, a.output, speaker, "va")
    for row in train:
        row["speaker"] = speaker
    for name, rows in (("train", train), ("validation", val)):
        with (a.output / f"{name}.jsonl").open("w", encoding="utf-8") as f:
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
    hours = sum(r["seconds"] for r in train) / 3600
    print(json.dumps({"train": len(train), "validation": len(val), "train_hours": round(hours, 2), "speaker": speaker}))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile urdu/make_init_checkpoint.py
"""Wrap the released inference weights as a starting point for `tiny_voca.train --init-from`.

The release (tiny-voca.pt) has the speech model only: no critics, optimizer or
mel head. train.py's --init-from wants a full training state, so this builds a
minimal one at step 0. The missing parts (mel head, critics) start fresh, which
is correct for a step-0 branch: critics are only used after --gan-start.

The phoneme embedding is grown from 76 English symbols to English + Urdu by
train.py's load_compatible (new rows start fresh). Run it through run.py so the
Urdu inventory is installed:

    python urdu/run.py make_init_checkpoint --weights weights/tiny-voca.pt --output runs/init.pt
"""

import argparse
import json
from pathlib import Path

import torch

from tiny_voca.config import CHECKPOINT_FORMAT


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--weights", type=Path, required=True)
    p.add_argument("--output", type=Path, required=True)
    a = p.parse_args()
    released = torch.load(a.weights, map_location="cpu", weights_only=True)
    if released.get("format_version") != CHECKPOINT_FORMAT:
        raise SystemExit(f"unexpected checkpoint format {released.get('format_version')}")
    from tiny_voca import config as config_module   # the Urdu-extended Config (installed by run.py)
    target = config_module.Config().to_dict()
    if {**released["config"], "vocab_size": target["vocab_size"]} != target:
        raise SystemExit("released config differs from tiny_voca.config.Config() defaults")
    state = {
        "format_version": CHECKPOINT_FORMAT, "config": target,
        "frontend": released["frontend"], "annotation": released.get("annotation"),
        "step": 0, "datasets": [], "recipe": {"gan_start": 0, "note": "English tiny-voca, inference weights only"},
        "system": {"model." + k: v for k, v in released["model"].items()},
    }
    a.output.parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, a.output)
    print(json.dumps({"tensors": len(released["model"]), "vocab": f"76 -> {target['vocab_size']}", "output": str(a.output)}))


if __name__ == "__main__":
    main()

## 1. Data: stream clips, resample to 24 kHz (skipped if already done)

In [ ]:
import os
def sh(cmd):
    import subprocess; subprocess.run(cmd, shell=True, check=True)
if not os.path.exists("data/validation.jsonl"):
    sh(f"python urdu/prepare_data.py --train {N_TRAIN} --validation {N_VAL} --output data")
else: print("data cached")

## 2. Acoustic features (skipped if already done)

In [ ]:
import json
for split in ("train", "validation"):
    meta = f"features/{split}.preparation.json"
    if os.path.exists(meta) and json.load(open(meta)).get("complete"):
        print(split, "features cached"); continue
    resume = "--resume" if os.path.exists(meta) else ""   # finish a half-done run instead of restarting
    sh(f"python urdu/run.py tiny_voca.prepare --inputs data/{split}.jsonl --split {split} --output features --speaker-column speaker {resume}")

## 3. Forced alignment (durations are cached per clip; re-running only re-applies the quality gate)
The default gate rejected 998/1000 real-speech clips (it is tuned for synthetic English), so the thresholds are relaxed here. Check the printout: `accepted` should be close to 1000.

In [ ]:
import json
GATE = "--max-weak-phone-fraction 0.8 --min-token-probability 0.1 --min-vowel-voiced 0.2 --max-phone-frames 120 --min-phones-per-second 3 --max-phones-per-second 35"
resume = "--resume" if os.path.exists("annotations-v2/annotations.json") else ""
sh(f"python urdu/run.py tiny_voca.annotate --features features --output annotations-v2 --splits train validation "
   f"--dataset-revision UrduTTS-streamed --device cuda {GATE} {resume}")
# annotate only marks a run 'complete' when it also saw a 'test' split; we don't need one.
meta = json.load(open("annotations-v2/annotations.json"))
counts = meta.get("counts") or {}
summary = json.load(open("annotations-v2/summary.json"))
for k, v in summary["splits"].items():
    print(k, "accepted", v["accepted"], "rejected", v["rejected"], v["rejection_reasons"])
assert summary["splits"]["train"]["accepted"] >= 200, "too few clips accepted; see qc_quantiles in annotations-v2/summary.json"
meta["complete"] = True
json.dump(meta, open("annotations-v2/annotations.json", "w"))

## 4. Fine-tune from English tiny-voca (resumes from runs/urdu2/last.pt if it exists)

In [ ]:
if not os.path.exists("runs/init-v2.pt"):
    sh("python urdu/run.py make_init_checkpoint --weights weights/tiny-voca.pt --output runs/init-v2.pt")
start = "--resume runs/urdu2/last.pt" if os.path.exists("runs/urdu2/last.pt") else "--init-from runs/init-v2.pt --new-data"
sh(f"python urdu/run.py tiny_voca.train --train annotations-v2/train.jsonl --validation annotations-v2/validation.jsonl "
   f"--output runs/urdu2 {start} --device cuda --steps {STEPS} --max-seconds 86400 --learning-rate 2e-4 --warmup 200 "
   f"--gan-start {STEPS//2} --gan-ramp {STEPS//8} --teacher-hold {STEPS//6} --teacher-ramp {STEPS//3} --teacher-final 0 "
   f"--save-every 1000 --keep-every 5000 --log-every 25 --validation-size {N_VAL}")

## 5. Speak Roman Urdu

In [ ]:
import json, subprocess, IPython.display as ipd
voice = json.loads(open("data/train.jsonl").readline())["speaker"]
for i, t in enumerate(["aap kaise hain?", "mujhe chai pasand hai.", "shukriya, bohat shukriya!"]):
    subprocess.run(["python","urdu/run.py","tiny_voca.infer","--checkpoint","runs/urdu2/inference.pt",
                    "--voice",voice,"--text",t,"--output",f"out{i}.wav","--device","cuda"],check=True)
    print(t); ipd.display(ipd.Audio(f"out{i}.wav"))

## 6. Export for the browser (ONNX)
Download `/kaggle/working/web/model.onnx` and `model.json` from the Output tab and put them in `web/public/` of the Next.js app.
**New session, no retraining:** Add Input -> Your Work -> this notebook's output, then run only the setup, `%cd`, `%%writefile` cells and this one.

In [ ]:
import glob, os, shutil, subprocess
subprocess.run("pip install -q onnx", shell=True)
ckpt = next(p for p in ["runs/urdu2/inference.pt", *glob.glob("/kaggle/input/**/runs/urdu2/inference.pt", recursive=True)] if os.path.exists(p))
print("checkpoint:", ckpt)
os.makedirs("/kaggle/working/web", exist_ok=True)
subprocess.run(["python","urdu/run.py","tiny_voca.export","--checkpoint",ckpt,"--output","/kaggle/working/web/model.onnx"], check=True)
for p in glob.glob("/kaggle/working/web/*"): print(p, round(os.path.getsize(p)/1e6, 2), "MB")